# Tutorial 3: Working with Your Own Data

This tutorial demonstrates how to apply econirl to your own datasets.
We'll cover the complete workflow from data preparation to interpretation.

## Overview

To estimate a dynamic discrete choice model, you need:

1. **Panel data**: Observations of states and actions over time for multiple individuals
2. **State space definition**: How to discretize your state variable
3. **Transition probabilities**: How states evolve (can be estimated from data)
4. **Utility specification**: What features enter the utility function

We'll walk through each step using a subset of the Rust bus data,
treating it as if it were your own dataset.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

from econirl import RustBusEnvironment, LinearUtility, NFXPEstimator
from econirl.core.types import Panel, DDCProblem
from econirl.datasets import load_rust_bus

print("econirl loaded successfully!")

## Step 1: Load and Prepare Your Data

Your data should have:
- An individual identifier
- A time/period variable
- A state variable (continuous or already discretized)
- An action variable (discrete choices)

In [ ]:
# Load example data (replace with your own data loading)
df = load_rust_bus(group=1)  # Just group 1 for this example

print(f"Data shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Check data structure
print("Data Summary")
print("=" * 50)
print(f"Individuals: {df['bus_id'].nunique()}")
print(f"Periods per individual: {df.groupby('bus_id')['period'].count().describe()}")
print(f"\nState variable (mileage_bin):")
print(f"  Range: [{df['mileage_bin'].min()}, {df['mileage_bin'].max()}]")
print(f"  Unique values: {df['mileage_bin'].nunique()}")
print(f"\nAction variable (replaced):")
print(f"  Values: {df['replaced'].unique()}")
print(f"  Frequency: {df['replaced'].value_counts().to_dict()}")

## Step 2: Define State Space

If your state is continuous, you'll need to discretize it.
The choice of bins involves a bias-variance tradeoff:
- More bins: Less approximation error, but more parameters to estimate
- Fewer bins: More approximation error, but more precise estimates

In [ ]:
# Define state space parameters
NUM_STATES = 90  # Number of mileage bins
NUM_ACTIONS = 2  # Keep (0) or Replace (1)

# If your data has continuous states, discretize them:
# df['state_bin'] = pd.cut(df['continuous_state'], bins=NUM_STATES, labels=False)

# Our data already has discretized states
print(f"State space size: {NUM_STATES}")
print(f"Action space size: {NUM_ACTIONS}")

## Step 3: Estimate Transition Probabilities

Transition probabilities describe how states evolve given actions.
These can be estimated directly from the data.

In [ ]:
def estimate_transitions(df, num_states, num_actions):
    """
    Estimate transition probabilities from panel data.
    
    Returns:
        transitions: tensor of shape (num_actions, num_states, num_states)
    """
    # Count transitions for each (action, state, next_state)
    transitions = np.zeros((num_actions, num_states, num_states))
    
    for bus_id in df['bus_id'].unique():
        bus_data = df[df['bus_id'] == bus_id].sort_values('period')
        
        states = bus_data['mileage_bin'].values
        actions = bus_data['replaced'].values
        
        for t in range(len(states) - 1):
            s = states[t]
            a = actions[t]
            s_next = states[t + 1]
            
            if s < num_states and s_next < num_states:
                transitions[a, s, s_next] += 1
    
    # Normalize to probabilities
    for a in range(num_actions):
        for s in range(num_states):
            total = transitions[a, s, :].sum()
            if total > 0:
                transitions[a, s, :] /= total
            else:
                # No observations: use uniform or prior
                transitions[a, s, :] = 1.0 / num_states
    
    return torch.tensor(transitions, dtype=torch.float32)

# Estimate transitions
transitions = estimate_transitions(df, NUM_STATES, NUM_ACTIONS)

print(f"Transition matrix shape: {transitions.shape}")
print(f"\nExample: P(s'|s=10, a=Keep):")
print(f"  Nonzero transitions: {(transitions[0, 10, :] > 0.01).sum()} states")

In [ ]:
# Visualize estimated transitions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for a, (ax, title) in enumerate(zip(axes, ['Keep', 'Replace'])):
    im = ax.imshow(transitions[a, :30, :30].numpy(), cmap='Blues', aspect='auto')
    ax.set_xlabel('Next State')
    ax.set_ylabel('Current State')
    ax.set_title(f'P(s\'|s, {title})')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## Step 4: Convert Data to Panel Format

econirl expects data in `Panel` format - a collection of trajectories.

In [ ]:
def dataframe_to_panel(df, state_col='mileage_bin', action_col='replaced', 
                       id_col='bus_id', time_col='period'):
    """
    Convert a pandas DataFrame to econirl Panel format.
    """
    trajectories = []
    
    for ind_id in df[id_col].unique():
        ind_data = df[df[id_col] == ind_id].sort_values(time_col)
        
        states = torch.tensor(ind_data[state_col].values, dtype=torch.long)
        actions = torch.tensor(ind_data[action_col].values, dtype=torch.long)
        
        trajectories.append((states, actions))
    
    return Panel.from_trajectories(trajectories)

# Convert our data
panel = dataframe_to_panel(df)

print(f"Panel created:")
print(f"  Individuals: {panel.num_individuals}")
print(f"  Total observations: {panel.num_observations:,}")

## Step 5: Define the Utility Specification

The utility specification defines what features enter the utility function.
For the bus problem:
- Keep: utility depends on mileage (maintenance cost increases)
- Replace: fixed cost

In [ ]:
def create_feature_matrix(num_states, num_actions):
    """
    Create feature matrix for linear utility.
    
    Features:
    - Feature 0: Mileage (for Keep action)
    - Feature 1: Constant (for Replace action)
    
    Shape: (num_states, num_actions, num_features)
    """
    num_features = 2
    features = torch.zeros(num_states, num_actions, num_features)
    
    for s in range(num_states):
        # Keep action: mileage enters utility
        features[s, 0, 0] = s  # Operating cost feature
        features[s, 0, 1] = 0  # No replacement cost
        
        # Replace action: fixed cost
        features[s, 1, 0] = 0  # No operating cost (reset to 0)
        features[s, 1, 1] = 1  # Replacement cost applies
    
    return features

# Create features
feature_matrix = create_feature_matrix(NUM_STATES, NUM_ACTIONS)
print(f"Feature matrix shape: {feature_matrix.shape}")

# Create utility specification
utility = LinearUtility(
    feature_matrix=feature_matrix,
    parameter_names=['operating_cost', 'replacement_cost']
)

print(f"Parameters to estimate: {utility.parameter_names}")

## Step 6: Set Up the Problem Specification

The `DDCProblem` contains structural parameters that are typically
assumed rather than estimated (discount factor, error distribution scale).

In [ ]:
# Create problem specification
problem = DDCProblem(
    num_states=NUM_STATES,
    num_actions=NUM_ACTIONS,
    discount_factor=0.9999,  # High discount (patient agent)
    scale_parameter=1.0,     # Normalize error variance
)

print(f"Problem Specification:")
print(f"  States: {problem.num_states}")
print(f"  Actions: {problem.num_actions}")
print(f"  Discount factor: {problem.discount_factor}")

## Step 7: Estimate the Model

In [ ]:
# Create estimator
estimator = NFXPEstimator(
    se_method="asymptotic",
    verbose=True,
)

# Run estimation
result = estimator.estimate(
    panel=panel,
    utility=utility,
    problem=problem,
    transitions=transitions,
)

print("\nEstimation complete!")

In [ ]:
# View results
print(result.summary())

## Step 8: Interpret Results

The estimated parameters have economic interpretations:

- **Operating cost**: Each mileage bin increases disutility by this amount.
  Higher values mean maintenance costs rise steeply with mileage.

- **Replacement cost**: Fixed disutility from replacing the engine.
  Higher values mean agents delay replacement longer.

In [ ]:
# Economic interpretation
theta_c = result.parameters[0].item()
RC = result.parameters[1].item()

print("Economic Interpretation")
print("=" * 50)
print(f"\nOperating cost (per mileage bin): {theta_c:.6f}")
print(f"  At mileage bin 50: utility cost = {50 * theta_c:.4f}")
print(f"  At mileage bin 80: utility cost = {80 * theta_c:.4f}")

print(f"\nReplacement cost: {RC:.4f}")

# Indifference point (roughly where replacement becomes attractive)
indiff_mileage = RC / theta_c if theta_c > 0 else float('inf')
print(f"\nApproximate indifference mileage: {indiff_mileage:.0f} bins")
print(f"  (This is a rough approximation; actual policy depends on dynamics)")

In [ ]:
# Plot estimated policy
policy = result.policy.numpy()

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(NUM_STATES), policy[:, 1], 'b-', linewidth=2)
ax.fill_between(range(NUM_STATES), 0, policy[:, 1], alpha=0.3)

ax.set_xlabel('Mileage Bin')
ax.set_ylabel('P(Replace)')
ax.set_title('Estimated Replacement Policy')
ax.set_xlim(0, 70)
ax.set_ylim(0, 0.3)

plt.tight_layout()
plt.show()

## Summary: Complete Workflow

To apply econirl to your own data:

1. **Prepare data**: Panel format with individual ID, time, state, action
2. **Discretize states** if needed (choose bins carefully)
3. **Estimate transitions** from the data
4. **Define utility features**: What affects choices?
5. **Set structural parameters**: Discount factor, error scale
6. **Estimate** using NFXP or other methods
7. **Interpret** parameters economically

### Common Issues

- **Sparse data**: Some state-action combinations rarely observed
  - Solution: Use fewer bins or regularization
  
- **Identification**: Some parameters may be hard to identify
  - Check: Hessian condition number in output
  
- **Convergence**: Optimization may be slow or fail
  - Try: Different starting values, different optimizers